# Data preparation notes

Work done on 16 November 2026. Notes added on 22 November 2026.

These notes cover the log conversion and the first checks. The data has not been cleaned yet.


## Project aim

The aim is to find possible detonation risk areas and possible other poor running condition areas from the existing Speeduino sensor data. There are no confirmed knock measurements in these logs. The model will not show whether knock actually happened.

Synthetic logs of fault situations will also be created. This allows testing changes such as rising AFR or a sudden increase in ignition advance without causing them in the car. The changes need to fit the engine operating conditions.

One planned comparison is raw sensor values against the same values with added time and interaction features. The model choices are still not decided.


## Original data

There are 17 MLG logs. The converted files have 50 columns each. The logs include engine starts, idle, driving, engine-off time and tests made during earlier fault finding.

There is less driving data than idle data. It is needed to check how much data is available at different RPM and MAP values before choosing the test cases.

The original MLG files are in `data/raw/`. Converted CSV files are in `data/csv/`. The Excel workbooks are in `data/xlsx/`.


## MLG to CSV conversion

The first converter is kept as `tools/mlg_to_csv_v1.py`. The corrected version is `tools/mlg_to_csv_v2.py`.

The converter reads the file header and the description of each field. These descriptions give the field name, type, units and scaling. They are then used to read each measurement row.

The file format is described in the EFI Analytics MLG specification. The measurement payload length comes from the header, so the converter does not assume that every MLG file has 50 fields.

In the current logs, the measurement payload is 91 bytes. A full measurement block is 96 bytes: four prefix bytes, the payload and one checksum byte.

Scalar values are calculated as `(raw_value + transform) * scale`. The current script also rounds values when the field's display precision is positive. It therefore exports scaled, display-rounded values rather than the original raw bytes.


## MARK handling and checks

The first version did not handle MARK blocks correctly. It skipped them as if they were measurement blocks. A MARK block is only 54 bytes, including its 50-byte comment. This moved the reading position to the wrong place and caused incorrect values after the marker.

This was visible in `night.mlg`. The real time jump from 629.369 s to 1197.157 s was also visible in MegaLogViewer. After that jump, the first converter produced time values around 1.685e22 s.

V2 handles MARK blocks separately. The real time gap remains, but the incorrect huge time values are gone. The corrected log reaches about 1425.268 s. The MARK comments include Going Offline, Going Online and RESET. These support the event of a connection loss and restart during that session caused by the monitor application reset.

V2 checks the sum of the payload bytes modulo 256 against the stored checksum. It also stops on unknown or incomplete blocks. All 17 supplied MLG files passed the checksum checks. Logs without MARK blocks gave the same DataFrames as V1. values have been compared from the corrected export with MegaLogViewer.

The checksum checks the stored bytes. It does not prove that a sensor reading is physically correct.


## Excel export

Importing the CSV files manually, Excel interpreted some decimal values as times. For example, 13.8 became 13:08 and then appeared as a fraction of a day when converted to a number.

Using Decimal Number with the English (United States) locale fixed the import when applied to the original text values. ISNUMBER alone was not enough to check the result because Excel times are also numbers.

The script `tools/csv_to_excel.py` now combines the CSV files into one workbook. Each file gets its own sheet. It converts the values to numbers before writing them, so Excel does not need to guess their type.

The export keeps all rows and columns in their original order. Repeated timestamps are kept. Missing values become empty Excel cells, not zeros. The workbook has 17 sheets and 50 columns per log. It also has filters, readable column widths and a frozen first row and column.


## Observations

The misfire logs were recorded during abnormal engine operation.

At 75.949 s in `misfire.csv`, Excel and MegaLogViewer showed matching values: RPM 1951, MAP 50 kPa, TPS 6%, AFR 17.7 and DFCO 0.

In my usual driving, AFR readings are normally below 14. Readings around 20 can also occur during engine braking. A high AFR reading alone is therefore not enough to label a row as risky or to confirm misfire. The operating conditions and related signals need to be checked together.

Some logs contain connection losses, resets and sync loss events. These are useful cases to examine, but should not automatically be treated as normal driving or as confirmed detonation risk. 


## Decisions for preprocessing

- Keeping the original MLG files and converted CSV files unchanged. Save cleaned data separately in `data/processed/`.
- Keeping repeated timestamps and the original row order. A repeated time does not mean the measurements are identical.
- Recording manual start and end limits before applying them with Python. Include the reason for each limit.
- Removing clear extra engine-off waiting time only after checking the context. Keep the start and shutdown events for now.
- Do not remove every RPM = 0 row. A short drop may be part of a sync loss event.
- Do not calculate rolling features or changes across long recording gaps. Split the data into continuous sections first.
- Keeping known abnormal logs marked separately. Do not assume the whole dataset describes normal operation.
- More planning of the usage of synthetic data.


## Next steps

1. List the row count, time range, gaps and missing values for each log.
2. Check the amount of engine-off time, idle and loaded operation.
3. Record clear start and end limits with reasons.
4. Apply the agreed limits to separate CSV files and compare before and after.
5. Add plots and short notes here as the analysis continues.

Both conversion scripts were reviewed line by line. Added comments to help explain their behaviour and to practice Python. Artificial intelligence has been leveraged in developing the conversion scripts and studying their functionality and possible problems. Every time AI is leveraged, it will be accordingly documented.


## Reference

P. Tobin, "Binary MLG Logging (MLVLG) file format specification," EFI Analytics, format version 2, revision 0.94, August 8, 2022. Accessed November 16, 2026.

https://www.efianalytics.com/TunerStudio/docs/MLG_Binary_LogFormat_2.0.pdf

Used for the header, field definitions, scaling, measurement and MARK block sizes, and checksum calculation. Research sources for engine risk factors and ML methods will be added when those parts are developed. The working source register is kept separately in the project for now.
